# Ответы на экзаменационные вопросы: обработка текста и хеширование строк

Ноутбук закрывает вопросы **21, 22**, а также практические задачи из билетов
**№24 (задача 1, теория дублирует вопрос 21)**, **№19 (задача 3)** и **№14 (задача 3)**.

Используемый датасет: `datasets/anna_synthetic.txt` — синтетический текст, сгенерированный в
`exam/make_datasets.py` (это **не** текст романа Л.Н. Толстого, а собственноручно сгенерированный
текст с главной героиней «Анна», созданный специально для демонстрации частот словоформ по
падежам и топ-N частотных слов).

In [1]:
import sys
import re
import time
from pathlib import Path
from collections import Counter
from functools import lru_cache

sys.path.insert(0, str(Path.cwd()))
import make_datasets

DATA = make_datasets.DATA_DIR
make_datasets.make_all()

import numpy as np
import pandas as pd
import pymorphy3
from nltk.stem.snowball import SnowballStemmer

pd.set_option("display.max_rows", 30)


Готово. Файлы в /Users/msikanov/PycharmProjects/ege-informatics/exam/datasets:
 - addres-book-q.xml
 - address_book_men.pkl
 - address_book_women.pkl
 - anna_synthetic.txt
 - countries-of-the-world.csv
 - demo_array.npy
 - demo_data.h5
 - demo_people.csv
 - demo_people.pkl
 - litw-win.csv
 - person.json
 - sp500hst.txt
 - sp500hst_yearly_avg.csv
 - titanic.csv
 - себестоимость_в1.xlsx
 - себестоимость_в1_recalculated.xlsx


## Вопрос 21 (= билет №24, задача 1). Сегментация и токенизация текста, стемминг и лемматизация

### Сегментация

Разбиение текста на более крупные смысловые единицы — **предложения** (segmentation по границам
`.`, `!`, `?`, с учётом исключений вроде сокращений) или абзацы. Первый шаг любой NLP-обработки,
после которого работают уже с отдельными предложениями.

### Токенизация

Разбиение предложения/текста на **токены** — слова, знаки пунктуации, числа. Простейший вариант —
разбить по пробелам и знакам пунктуации регулярным выражением; более аккуратные токенизаторы
учитывают сокращения, дефисные слова, знаки внутри чисел и т.п.

### Стемминг

Грубое, **эвристическое** усечение слова до "основы" (stem) путём отсечения окончаний/суффиксов по
формальным правилам, **без обращения к словарю**. Результат может не быть настоящим словом языка,
но одинаков для разных словоформ одного корня. Быстро, но не всегда точно (ошибки "недостемминга" и
"перестемминга").

### Лемматизация

Приведение словоформы к **словарной начальной форме** (лемме) — например, для существительного это
именительный падеж единственного числа, для глагола — инфинитив. В отличие от стемминга, требует
морфологического словаря/анализатора (учитывает часть речи и грамматические категории), поэтому
точнее, но медленнее. Для русского языка — библиотека `pymorphy3` (по словарю OpenCorpora).

In [2]:
paragraph = (
    "Анна долго молчала, глядя в окно на засыпанный снегом сад. "
    "Она думала о письмах, о доме и о разговоре с друзьями! "
    "Гости уже собирались в зале, а слуги расставляли чашки на столе?"
)

# --- сегментация: разбиваем текст на предложения по знакам конца предложения ---
sentences = re.split(r"(?<=[.!?])\s+", paragraph.strip())
print("Сегментация на предложения:")
for s in sentences:
    print(" -", s)

# --- токенизация: разбиваем предложение на слова-токены ---
tokens = re.findall(r"[а-яё]+", sentences[0].lower())
print("\nТокенизация первого предложения:", tokens)

# --- стемминг vs лемматизация ---
stemmer = SnowballStemmer("russian")
morph = pymorphy3.MorphAnalyzer()

words = ["домах", "домом", "дома", "домик", "думала", "думать", "думают"]
print(f"\n{'слово':<10}{'стемминг':<12}{'лемматизация'}")
for w in words:
    stem = stemmer.stem(w)
    lemma = morph.parse(w)[0].normal_form
    print(f"{w:<10}{stem:<12}{lemma}")


Сегментация на предложения:
 - Анна долго молчала, глядя в окно на засыпанный снегом сад.
 - Она думала о письмах, о доме и о разговоре с друзьями!
 - Гости уже собирались в зале, а слуги расставляли чашки на столе?

Токенизация первого предложения: ['анна', 'долго', 'молчала', 'глядя', 'в', 'окно', 'на', 'засыпанный', 'снегом', 'сад']

слово     стемминг    лемматизация
домах     дом         дом
домом     дом         дом
дома      дом         дом
домик     домик       домик
думала    дума        думать
думать    дума        думать
думают    дума        думать


## Вопрос 22. Хеширование строк для ускорения операций, ограничения, параметризация

### Зачем хешировать строки

Хеш-функция `h(s)` отображает строку произвольной длины в число фиксированного размера
(например, 64-битное). Это ускоряет типичные операции со строками:

- **проверка на равенство/поиск в множестве** — вместо посимвольного сравнения `O(n)` со всеми
  элементами структуры данных, строки помещают в `dict`/`set` (хеш-таблицу): вычислили хеш один
  раз — и далее сравнение сводится к сравнению чисел и поиску в корзине хеш-таблицы, в среднем
  `O(1)` вместо `O(n)` по количеству элементов;
- **быстрое "предсравнение" двух длинных строк**: если хеши разные — строки точно разные (без
  посимвольного сравнения); если хеши совпали — тогда (и только тогда) стоит сравнить посимвольно,
  чтобы исключить редкую коллизию;
- **скользящий (rolling) хеш** — пересчитывать хеш подстроки при сдвиге окна на 1 символ за
  `O(1)`, а не пересчитывать с нуля за `O(m)` — основа алгоритма Рабина–Карпа для поиска подстрок
  и сравнения больших текстовых блоков.

### Хеширование строк в Python

Встроенная `hash(s)` для строк использует алгоритм **SipHash** со случайным ключом (`PYTHONHASHSEED`),
который генерируется заново при каждом запуске интерпретатора (защита от DoS-атак через
"hash-flooding" — умышленный подбор строк-коллизий для конкретной хеш-функции). Из-за этого:

- `hash("abc")` **отличается между разными запусками** процесса Python (если не зафиксировать
  `PYTHONHASHSEED`);
- хеш нельзя сохранять на диск и переиспользовать между запусками — только в пределах одного
  процесса;
- гарантия: равные строки → равные хеши **в пределах одного запуска**; но не наоборот.

### Математические ограничения

- **Принцип Дирихле (pigeonhole)**: множество строк бесконечно (или очень велико), а хеш —
  число фиксированной разрядности (например, 64 бита → `2^64` значений). Значит **коллизии
  неизбежны** — не существует инъективной хеш-функции для строк произвольной длины в конечный
  диапазон.
- **Оценка "дня рождения" (birthday bound)**: при случайной хеш-функции с диапазоном `N` значений
  первая коллизия ожидается уже при `~√N` хешированных строках, а не при `~N` — вероятность
  коллизии растёт значительно быстрее интуиции.
- Отсюда вывод: любое сравнение "по хешу" **необходимо**, но **не достаточно** — при совпадении
  хешей нужна дополнительная проверка (полное сравнение строк), если результат должен быть строго
  корректным.

### Параметризация хеширования для сравнения строк

Для полиномиального (rolling) хеша `h(s) = (s[0]·b^(m-1) + s[1]·b^(m-2) + ... + s[m-1]) mod M`
результат и вероятность коллизии зависят от выбора параметров:

- **основание `b`** — обычно простое число, желательно больше размера алфавита;
- **модуль `M`** — большое простое число: чем больше `M`, тем меньше вероятность случайной
  коллизии (`~1/M` для двух случайных строк), но тем больше сама хеш-величина;
- **двойное хеширование** (два независимых `(b, M)`, сравнение по паре хешей) — снижает
  вероятность ложной коллизии до `~1/(M₁·M₂)`, то есть перемножает надёжность.

In [3]:
# --- 1) hash() для строк, ускорение поиска через set/dict (в среднем O(1)) ---
words_pool = [f"слово_{i}" for i in range(200_000)]
target = "слово_199999"

t0 = time.perf_counter()
found_linear = target in words_pool          # O(n): линейный просмотр списка
t1 = time.perf_counter()

words_set = set(words_pool)                    # построение хеш-множества (один раз)
t2 = time.perf_counter()
found_hash = target in words_set                # O(1) в среднем: по хешу строки сразу находим корзину
t3 = time.perf_counter()

print(f"поиск в списке (O(n)):      {t1 - t0:.6f} c, найдено: {found_linear}")
print(f"поиск в set (в среднем O(1)): {t3 - t2:.6f} c, найдено: {found_hash}")
print(f"ускорение поиска: {(t1 - t0) / (t3 - t2):.0f}x")

print("\nhash() у равных строк совпадает в пределах одного запуска:", hash("привет") == hash("привет"))


поиск в списке (O(n)):      0.001448 c, найдено: True
поиск в set (в среднем O(1)): 0.000023 c, найдено: True
ускорение поиска: 64x

hash() у равных строк совпадает в пределах одного запуска: True


In [4]:
# --- 2) собственный полиномиальный (rolling) хеш и коллизии при "плохом" модуле ---
def polynomial_hash(s: str, base: int, mod: int) -> int:
    h = 0
    for ch in s:
        h = (h * base + ord(ch)) % mod
    return h


sample_strings = [f"текст_{i}" for i in range(2000)]

# "плохой" (маленький) модуль -> коллизии почти гарантированы (принцип Дирихле)
small_mod_hashes = [polynomial_hash(s, base=131, mod=97) for s in sample_strings]
collisions_small = len(sample_strings) - len(set(small_mod_hashes))

# "хороший" (большой простой) модуль -> коллизии практически не встречаются на этой выборке
big_mod_hashes = [polynomial_hash(s, base=131, mod=2_147_483_647) for s in sample_strings]
collisions_big = len(sample_strings) - len(set(big_mod_hashes))

print(f"строк: {len(sample_strings)}")
print(f"коллизий при малом модуле (mod=97):              {collisions_small}")
print(f"коллизий при большом простом модуле (mod=2^31-1): {collisions_big}")


строк: 2000
коллизий при малом модуле (mod=97):              1903
коллизий при большом простом модуле (mod=2^31-1): 0


In [5]:
# --- 3) алгоритм Рабина-Карпа: сравнение подстрок за O(1) на сдвиг через rolling hash ---
def rabin_karp_find(text: str, pattern: str, base: int = 131, mod: int = 2_147_483_647):
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []

    pow_m1 = pow(base, m - 1, mod)
    pattern_hash = polynomial_hash(pattern, base, mod)

    window_hash = polynomial_hash(text[:m], base, mod)
    positions = []
    if window_hash == pattern_hash and text[:m] == pattern:   # проверка после совпадения хешей!
        positions.append(0)

    for i in range(1, n - m + 1):
        # rolling: убрать вклад ушедшего символа, сдвинуть, добавить новый - за O(1), без пересчёта с нуля
        window_hash = ((window_hash - ord(text[i - 1]) * pow_m1) * base + ord(text[i + m - 1])) % mod
        if window_hash == pattern_hash and text[i:i + m] == pattern:
            positions.append(i)
    return positions


anna_text = (make_datasets.DATA_DIR / "anna_synthetic.txt").read_text(encoding="utf-8")
positions = rabin_karp_find(anna_text.lower(), "анна")
print(f"вхождений точной подстроки 'анна' (алгоритм Рабина-Карпа): {len(positions)}")
print("первые 5 позиций:", positions[:5])
print("совпадает с count() стандартной библиотеки:", anna_text.lower().count("анна") == len(positions))


вхождений точной подстроки 'анна' (алгоритм Рабина-Карпа): 149
первые 5 позиций: [255, 462, 963, 2067, 2323]
совпадает с count() стандартной библиотеки: True


## Практика: билет №19, задача 3

> Для имени собственного главной героини найти частоту различных форм склонения имени по падежам,
> вывести словоформы и их частоты. Использовать лемматизацию. Рекомендация: для ускорения учесть
> длину слов и использовать мемоизацию.

Героиню зовут **Анна** — её словоформы по падежам: Анна (им.), Анны (род.), Анне (дат./предл.),
Анну (вин.), Анной (твор.). Используем `pymorphy3` для лемматизации токенов и относим к имени
только те слова, у которых **среди всех разборов** встречается лемма `"анна"` — это отличает,
например, «Анне» (падежная форма имени «Анна») от случайного слова, не связанного с именем.

Для ускорения: 1) длина словоформ имени лежит в диапазоне 4–5 символов — не прогоняем через
морфологический анализатор слова другой длины; 2) `functools.lru_cache` мемоизирует разбор —
каждое уникальное слово анализируется только один раз, сколько бы раз оно ни встретилось в тексте.

In [6]:
TARGET_LEMMA = "анна"
CASE_NAMES = {
    "nomn": "именительный", "gent": "родительный", "datv": "дательный",
    "accs": "винительный", "ablt": "творительный", "loct": "предложный",
}


@lru_cache(maxsize=None)
def name_case_if_matches(word_lower: str):
    if not (4 <= len(word_lower) <= 5):          # быстрый отсев по длине слова, без морфоанализа
        return None
    for p in morph.parse(word_lower):
        if p.normal_form == TARGET_LEMMA:
            return p.tag.case
    return None


tokens = re.findall(r"[а-яё]+", anna_text.lower())
print("всего токенов в тексте:", len(tokens), "| уникальных:", len(set(tokens)))

t0 = time.perf_counter()
wordform_counts = Counter()
case_counts = Counter()
for w in tokens:
    case = name_case_if_matches(w)
    if case is not None:
        wordform_counts[w] += 1
        case_counts[case] += 1
t1 = time.perf_counter()

print(f"обработка заняла {t1 - t0:.3f} c (кэш морфоанализа: {name_case_if_matches.cache_info()})")

print("\nЧастоты словоформ имени «Анна»:")
for form, cnt in wordform_counts.most_common():
    print(f"  {form:<8} — {cnt}")

print("\nЧастоты по падежам:")
for case, cnt in case_counts.most_common():
    print(f"  {CASE_NAMES.get(case, case):<14} — {cnt}")


всего токенов в тексте: 10832 | уникальных: 277
обработка заняла 0.004 c (кэш морфоанализа: CacheInfo(hits=10555, misses=277, maxsize=None, currsize=277))

Частоты словоформ имени «Анна»:
  анна     — 149
  анне     — 95
  анной    — 94
  анны     — 92
  анну     — 42

Частоты по падежам:
  именительный   — 149
  дательный      — 95
  творительный   — 94
  родительный    — 92
  винительный    — 42


**Замечание про форму «Анне».** У имени «Анна» дательный и предложный падеж совпадают по форме
(«к Анне» — дательный, «об Анне» — предложный). `pymorphy3` возвращает несколько разборов с
одинаковой вероятностью (`datv` и `loct`) и не может формально различить их без контекста
(синтаксического анализа предложения); в коде выше в такой ситуации фиксированно берётся первый
разбор из списка. Поэтому все встретившиеся «Анне» здесь показаны только по одному из двух
падежей — это ограничение морфологического анализа вне контекста предложения, а не ошибка счёта.

## Практика: билет №14, задача 3

> Определить 200 самых частотных слов в тексте (вывести слова и частоты). Среди них определить
> слова, не входящие в стандартный список стоп-слов, и вывести их на экран.

Список стоп-слов (частицы, союзы, предлоги, местоимения) задан вручную — без обращения к
корпусам `nltk.download(...)`, поскольку сетевой доступ к внешним хранилищам данных недоступен.

In [7]:
STOPWORDS = set(
    "и в не на я быть он с что а это весь как она по но они к у ты из мы за то "
    "свой который год у же вы за бы по только ей мне было вот от меня ещё нет о "
    "из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас "
    "нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть "
    "надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе "
    "под будет ж тогда кто этот того потому этого какой совсем ним здесь "
    "этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда "
    "можно при наконец два об другой хоть после над больше тот через эти нас "
    "про всего них какая много разве три эту моя впрочем хорошо свою этой "
    "перед иногда лучше чуть том нельзя такой им более всегда конечно всю "
    "между".split()
)

counts = Counter(tokens)
top200 = counts.most_common(200)

top200_df = pd.DataFrame(top200, columns=["слово", "частота"])
print(f"всего уникальных слов в тексте: {len(counts)}; показываем топ-{len(top200)}:")
print(top200_df.to_string(index=False))

non_stopwords = [(w, c) for w, c in top200 if w not in STOPWORDS]
print(f"\nиз топ-{len(top200)} слов НЕ входят в список стоп-слов ({len(non_stopwords)} слов):")
non_stop_df = pd.DataFrame(non_stopwords, columns=["слово", "частота"])
print(non_stop_df.to_string(index=False))


всего уникальных слов в тексте: 277; показываем топ-200:
          слово  частота
              и      369
              в      353
             на      324
           анна      149
              с      148
             не      133
          долго      102
             от       99
           анне       95
          анной       94
              у       93
           анны       92
          никто       91
          столе       84
              к       82
            сад       77
        чувство       75
           окно       75
         письмо       73
       разговор       72
            чай       68
        помещик       68
          гости       66
           поле       65
           часы       60
           утро       59
          вальс       59
          дождь       55
           дети       55
         играли       55
             во       55
          дворе       55
           пока       55
       взрослые       55
           пили       55
        террасе       55
       написала   